# Qualidade do Ar: Projete (conclua) e Implemente seu Produto

Bem-vindo ao laboratório final deste projeto. Aqui, novamente, você trabalhará com o conjunto de dados com o qual já se familiarizou da rede de monitoramento da qualidade do ar em Bogotá [RMCAB](http://201.245.192.252:81/home/map). Neste caderno, você concluirá as seguintes etapas:

1. Importe pacotes Python
2. Carregue o conjunto de dados com os valores ausentes preenchidos (resultado do último laboratório)
3. Use o método do vizinho mais próximo para fazer um mapa das PM2,5 em Bogotá
4. Teste diferentes valores de k para o método do vizinho mais próximo
5. Use o melhor valor de k para fazer um mapa de PM2,5 em Bogotá
6. Construa uma animação de mapa do PM2.5 em Bogotá
7. Exiba a animação do seu mapa

## 1. Importar pacotes Python

Execute a próxima célula para importar os pacotes Python necessários para este laboratório.

Observe a linha `import utils`. Esta linha importa as funções que foram escritas especificamente para este laboratório. Se você quiser ver quais são essas funções, vá em `Arquivo -> Abrir...` e abra o arquivo `utils.py` para dar uma olhada.

In [1]:
# Importar pacotes
import folium # Animações
import folium.plugins as plugins # Animações extras
import pandas as pd # Manipular dados
from sklearn.neighbors import KNeighborsRegressor # Pacote KNN
from datetime import datetime # Manipular datas/horas

import utiliTariosFaseC # Ferramentas próprias

print("Todos pacotes importados com sucesso!")

Todos pacotes importados com sucesso!


## 2. Carregue o conjunto de dados com os valores ausentes preenchidos (resultado do último laboratório)

Execute a próxima célula para ler o conjunto de dados que foi o resultado final do último laboratório, ou seja, um conjunto de dados com todos os valores faltantes para os poluentes preenchidos.

In [14]:
# Carregar o conjunto de dados, usando a primeira coluna como índice
full_dataset = pd.read_csv('data/DadosCompletosFaseA.csv', index_col=0)

# Converter a coluna DateTime para datetime
full_dataset['DateTime'] = pd.to_datetime(full_dataset['DateTime'], errors='coerce', dayfirst=True)

# Visualizar as primeiras linhas
full_dataset.head(5)


,DateTime,Station,Latitude,Longitude,PM2.5,PM10,NO,NO2,NOX,CO,OZONE,PM2.5_imputed_flag,PM10_imputed_flag,NO_imputed_flag,NO2_imputed_flag,NOX_imputed_flag,CO_imputed_flag,OZONE_imputed_flag
0,2021-01-01 00:00:00,USM,4.532056,-74.117139,32.7,56.6,7.504,15.962,23.493,0.44924,2.431,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-01-01 01:00:00,USM,4.532056,-74.117139,39.3,59.3,16.560,17.866,34.426,0.69832,1.121,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-01-01 02:00:00,USM,4.532056,-74.117139,70.8,96.4,22.989,17.802,40.791,0.88243,1.172,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021-01-01 03:00:00,USM,4.532056,-74.117139,81.0,108.3,3.704,9.886,13.591,0.29549,6.565,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-01-01 04:00:00,USM,4.532056,-74.117139,56.1,87.7,2.098,9.272,11.371,0.16621,9.513,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Use o método do vizinho mais próximo para fazer um mapa de PM2,5 em Bogotá
Aqui você usa o método do vizinho mais próximo para estimar os valores dos poluentes nos pontos entre as estações, para poder criar um bom mapa visual da poluição.

In [15]:
# Define a value for k (Vizinhos)
k = 3
#
# Definir o target do polutante
target = 'PM2.5'
#
# Defina um tamanho de célula da grade (valor mais alto implica uma grade mais fina)
n_points_grid = 68
neighbors_model = KNeighborsRegressor(n_neighbors=k, weights = 'distance', metric='sqeuclidean')
# 
#Isole uma única etapa de tempo do conjunto de dados
time_step = datetime.fromisoformat('2021-04-05T08:00:00')
time_step_data = full_dataset[full_dataset['DateTime'] == time_step]
neighbors_model.fit(time_step_data[['Latitude', 'Longitude']], time_step_data[[target]])
# Generate a map of predictions for Bogotá
predictions_xy, dlat, dlon = utiliTariosFaseC.predict_on_bogota(neighbors_model, n_points_grid)
utiliTariosFaseC.create_heat_map(predictions_xy, time_step_data, dlat, dlon, target)

/media/maykon/Arquivos/4-ProjetoQualidadeDoAr/venvQualidadeAr/lib/python3.10/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
/media/maykon/Arquivos/4-ProjetoQualidadeDoAr/venvQualidadeAr/lib/python3.10/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
/media/maykon/Arquivos/4-ProjetoQualidadeDoAr/venvQualidadeAr/lib/python3.10/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
/media/maykon/Arquivos/4-ProjetoQualidadeDoAr/venvQualidadeAr/lib/python3.10/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
/media/maykon/Arquivos/4-ProjetoQualidadeDoAr/venvQualidadeAr/lib/py

## 4. Teste diferentes valores de k para o método do vizinho mais próximo
Execute as células abaixo para calcular primeiro o erro médio absoluto (MAE) para k=1, ou em outras palavras, o erro associado ao uso de apenas um vizinho mais próximo, como você fez para criar o mapa acima. Depois disso, você executará o mesmo cálculo para diferentes valores de k.

A maneira como você está fazendo isso é semelhante ao que fez no laboratório anterior, onde calculou o MAE para usar medições de estações de sensores próximas para estimar o valor de PM2,5 em qualquer local de estação de sensores. Aqui você avaliará o método mostrado no mapa acima em cada localização de estação de sensor como se a medição dessa estação fosse substituída por um valor da estação vizinha mais próxima e, em seguida, uma média ponderada de k estações vizinhas mais próximas.


O cálculo do erro médio absoluto que está sendo executado pelo código anterior é o seguinte:

$$MAE = \frac{1}{n} \sum_{i=1}^{n}{|\rm{real}_i - \rm{modelo}_i|}$$
    
Onde "n" é o número de amostras no conjunto de dados de teste

In [11]:
# Faça uma estimativa do erro médio absoluto para k=1
utiliTariosFaseC.calculate_mae_for_k(full_dataset, k=1, target_pollutant=target)

7.990951687085921

After testing k=1, run the following cell to test a range of values for k.

In [12]:
# Faça uma estimativa do erro médio absoluto (MAE) para um intervalo de valores k.
kmin = 1
kmax = 7

for kneighbors in range(kmin, kmax+1):
    mae = utiliTariosFaseC.calculate_mae_for_k(full_dataset, k=kneighbors, target_pollutant=target)
    print(f'k = {kneighbors}, MAE = {mae}')

k = 1, MAE = 7.990951687085921
k = 2, MAE = 7.3784360503014055
k = 3, MAE = 7.237339373651147
k = 4, MAE = 7.192974827013243
k = 5, MAE = 7.1831419667215455
k = 6, MAE = 7.179474879422885
k = 7, MAE = 7.179023423730082


## 5. Use o melhor valor de k para fazer um mapa de PM2,5 em Bogotá

Execute a célula abaixo para gerar um mapa de valores PM2,5. O mapa mostrará a concentração do poluente escolhido sobre a cidade na `data_final` selecionada. Ao clicar nos círculos do mapa (estações), aparecem gráficos pop-up, mostrando a concentração do poluente no intervalo de tempo selecionado (de `data_inicial` a `data_final`). Você pode alterar os valores de datas e horas ou ` k` para ver como os dados diferem em vários momentos e como o resultado muda dependendo de `k`.

In [13]:
k = 3
start_date = datetime.fromisoformat('2021-08-02T08:00:00')
end_date = datetime.fromisoformat('2021-08-05T08:00:00')

utiliTariosFaseC.create_heat_map_with_date_range(full_dataset, start_date, end_date, k, target)

/media/maykon/Arquivos/4-ProjetoQualidadeDoAr/venvQualidadeAr/lib/python3.10/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(


ZeroDivisionError: float division by zero

## 6. Construa uma animação de mapa de PM2.5 em Bogotá
Execute a próxima célula para gerar uma animação de PM2.5 em um intervalo de tempo específico. Você pode alterar k para usar um número diferente de vizinhos e alterar as datas e horas para observar um intervalo de tempo diferente.

In [17]:
#Escolha os parâmetros da animação
import utiliTariosFaseC # Ferramentas próprias
k = 3
n_points_grid = 64
#
#Filtrar um período
start_date = datetime.fromisoformat('2021-08-04T08:00:00')
end_date = datetime.fromisoformat('2021-08-05T08:00:00')

# Crie os recursos para a animação (essas são as formas que aparecerão no mapa)
features = utiliTariosFaseC.create_animation_features(full_dataset, start_date, end_date, k, n_points_grid, target)
print('Recursos para a animação criada com sucesso! Execute a próxima célula para ver o resultado!')

AttributeError: module 'utiliTariosFaseC' has no attribute 'create_animation_features'

## 7. Display your map animation

Run the next cell to display the animation you created!

In [ ]:
# Create the map animation using the folium library
map_animation = folium.Map(location=[4.7110, -74.0721], zoom_start = 11) 
# Add the features to the animation
plugins.TimestampedGeoJson(
    {"type": "FeatureCollection", "features": features},
    period="PT1H",
    duration='PT1H',
    add_last_point=True
).add_to(map_animation)

# Run the animation
map_animation

## **Congratulations on finishing this lab!**

**Keep up the good work :)**